# Data Scientist (итерация 1)

# Data Scientist Report

Цель: построить воспроизводимый ML-ноутбук для детекции мошеннических вакансий (`fraudulent`) с акцентом на F1 и recall класса 1 при контроле precision.

План:
- загрузить очищенный датасет;
- сделать stratified split на train/val/test;
- построить осмысленные engineered features на основе trust, categorical и text-like полей;
- обучить baseline-модели;
- подобрать гиперпараметры для лучшей модели;
- честно оценить на test;
- сравнить с предыдущим best и сохранить артефакты.

In [ ]:
import os
import json
import re
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    average_precision_score, confusion_matrix, classification_report,
    precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.base import clone

import plotly.express as px
import plotly.graph_objects as go

FIGS = []
DATA_PATH = '/Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/cleaned.csv'
FEATURES_PATH = '/Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/features.csv'
MODEL_PATH = '/Users/iuriipostnii/Desktop/ГП3/gp3/data/memory/best_model.pkl'
METRICS_PATH = '/Users/iuriipostnii/Desktop/ГП3/gp3/data/memory/best_metrics.json'
TARGET = 'fraudulent'
RANDOM_STATE = 42

DF = pd.read_csv(DATA_PATH)
print('DF shape:', DF.shape)
print('Columns:', list(DF.columns))
print(DF[TARGET].value_counts(dropna=False))

DF shape: (17880, 17)
Columns: ['job_id', 'title', 'location', 'department', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']
fraudulent
0    17014
1      866
Name: count, dtype: int64


## Train/val/test split

Делаем стратифицированное разбиение, чтобы сохранить исходный дисбаланс классов на всех подвыборках. Решения по выбору модели и порога принимаются только по validation, финальная оценка — только по test.

In [ ]:
X = DF.drop(columns=[TARGET]).copy()
y = DF[TARGET].astype(int).copy()

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=RANDOM_STATE
)

print('Train:', X_train.shape, y_train.mean())
print('Val  :', X_val.shape, y_val.mean())
print('Test :', X_test.shape, y_test.mean())

pos = y_train.mean()
scale_pos_weight = max((1 - pos) / max(pos, 1e-9), 1.0)
print('Approx scale_pos_weight:', round(scale_pos_weight, 4))

Train: (10728, 16) 0.048471290082028336
Val  : (3576, 16) 0.04837807606263982
Test : (3576, 16) 0.04837807606263982
Approx scale_pos_weight: 19.6308


## Feature engineering

Ниже создаются признаки, опирающиеся на EDA и бизнес-логику:

1. **Trust-фичи**: `has_company_logo`, `has_questions` и их взаимодействия — это сильные маркеры качества вакансии.
2. **Text-derived признаки** по `company_profile`, `description`, `requirements`, `benefits`, `title`:
   - число слов;
   - число символов;
   - флаг `missing`/placeholder;
   - флаг короткого текста.
3. **High-risk category flags** для потенциально рискованных категорий (`function`, `required_education`, `required_experience`, `employment_type`).
4. **Location/title heuristics**: наличие удалёнки, URL/email/телефона в описании, ALL CAPS и т.д.

Полный датасет с фичами сохраняется отдельно.

In [ ]:
def _safe_text(s):
    return s.fillna('missing').astype(str).str.strip()

def add_features(df):
    df = df.copy()

    for col in ['company_profile', 'description', 'requirements', 'benefits', 'title']:
        if col in df.columns:
            txt = _safe_text(df[col]).str.lower()
            df[f'{col}_char_count'] = txt.str.len()
            df[f'{col}_word_count'] = txt.str.split().str.len()
            df[f'{col}_is_missing_token'] = txt.isin(['missing', '', 'nan', 'none']).astype(int)
            df[f'{col}_is_short'] = (df[f'{col}_word_count'] <= 5).astype(int)

    if 'company_profile' in df.columns:
        cp = _safe_text(df['company_profile']).str.lower()
        df['company_profile_placeholder_flag'] = cp.str.contains(
            r'missing|not available|n/a|na|tbd|confidential|to be discussed', regex=True
        ).astype(int)

    if 'has_company_logo' in df.columns and 'has_questions' in df.columns:
        df['logo_and_questions'] = ((df['has_company_logo'] == 1) & (df['has_questions'] == 1)).astype(int)
        df['no_logo_and_no_questions'] = ((df['has_company_logo'] == 0) & (df['has_questions'] == 0)).astype(int)
        df['logo_x_questions'] = df['has_company_logo'].astype(int) * df['has_questions'].astype(int)

    if 'telecommuting' in df.columns:
        df['telecommuting_flag'] = df['telecommuting'].astype(int)

    if 'title' in df.columns:
        title = _safe_text(df['title'])
        df['title_has_seniority'] = title.str.lower().str.contains(r'senior|sr\.?|lead|junior|jr\.?|manager|director').astype(int)
        df['title_all_caps_ratio'] = title.apply(lambda x: sum(ch.isupper() for ch in x) / max(len(x), 1))

    text_sources = [c for c in ['description', 'requirements', 'benefits', 'company_profile'] if c in df.columns]
    for col in text_sources:
        txt = _safe_text(df[col]).str.lower()
        df[f'{col}_has_url'] = txt.str.contains(r'http|www\.|\.com|\.net|\.org', regex=True).astype(int)
        df[f'{col}_has_email'] = txt.str.contains(r'[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}', case=False, regex=True).astype(int)
        df[f'{col}_has_phone'] = txt.str.contains(r'\+?\d[\d\s\-\(\)]{6,}', regex=True).astype(int)
        df[f'{col}_has_money_terms'] = txt.str.contains(r'bonus|commission|income|earn|salary|paid|payment|usd|\$', regex=True).astype(int)

    high_risk_rules = {
        'function': ['administrative'],
        'required_education': ['high school or equivalent', 'certification'],
        'required_experience': ['entry level', 'internship'],
        'employment_type': ['part-time', 'temporary', 'contract']
    }
    for col, risky_values in high_risk_rules.items():
        if col in df.columns:
            base = _safe_text(df[col]).str.lower()
            for val in risky_values:
                clean_name = re.sub(r'[^a-z0-9]+', '_', val).strip('_')
                df[f'{col}__is_{clean_name}'] = (base == val).astype(int)

    return df

X_train_fe = add_features(X_train)
X_val_fe = add_features(X_val)
X_test_fe = add_features(X_test)
X_train_full_fe = add_features(X_train_full)
X_all_fe = add_features(X)

feature_df_to_save = pd.concat([
    X_train_fe.assign(_split='train', fraudulent=y_train.values),
    X_val_fe.assign(_split='val', fraudulent=y_val.values),
    X_test_fe.assign(_split='test', fraudulent=y_test.values)
], axis=0)
feature_df_to_save.to_csv(FEATURES_PATH, index=False)

print('Engineered train shape:', X_train_fe.shape)
print('Saved features to:', FEATURES_PATH)
print('New columns example:', [c for c in X_train_fe.columns if c not in X_train.columns][:20])

Engineered train shape: (10728, 67)
Saved features to: /Users/iuriipostnii/Desktop/ГП3/gp3/data/processed/features.csv
New columns example: ['company_profile_char_count', 'company_profile_word_count', 'company_profile_is_missing_token', 'company_profile_is_short', 'description_char_count', 'description_word_count', 'description_is_missing_token', 'description_is_short', 'requirements_char_count', 'requirements_word_count', 'requirements_is_missing_token', 'requirements_is_short', 'benefits_char_count', 'benefits_word_count', 'benefits_is_missing_token', 'benefits_is_short', 'title_char_count', 'title_word_count', 'title_is_missing_token', 'title_is_short']


## Baseline-модели

Собираем единый preprocessing pipeline:
- numeric → impute + scale для линейной модели;
- categorical/text-like → impute + one-hot.

Сравниваем несколько baseline-моделей:
- `LogisticRegression(class_weight='balanced')`;
- `RandomForestClassifier(class_weight='balanced')`;
- `GradientBoostingClassifier`.

Дополнительно подбираем порог на validation по precision-recall curve: берём лучший F1 при бизнес-ограничении `precision >= 0.5`, а если таких точек нет — глобально лучший F1.

In [ ]:
def split_columns(df):
    categorical_cols = []
    numeric_cols = []
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_cols.append(col)
        else:
            categorical_cols.append(col)
    return numeric_cols, categorical_cols

num_cols, cat_cols = split_columns(X_train_fe)
print('Numeric cols:', len(num_cols))
print('Categorical cols:', len(cat_cols))

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=10))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat', categorical_transformer, cat_cols)
])

def find_best_threshold(y_true, probas, min_precision=0.5):
    precision, recall, thresholds = precision_recall_curve(y_true, probas)
    candidates = []
    for i, thr in enumerate(thresholds):
        p = precision[i]
        r = recall[i]
        f1 = 0.0 if (p + r) == 0 else 2 * p * r / (p + r)
        candidates.append((thr, p, r, f1))
    if not candidates:
        return 0.5, {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
    filtered = [x for x in candidates if x[1] >= min_precision]
    chosen = max(filtered if filtered else candidates, key=lambda x: x[3])
    return chosen[0], {'precision': chosen[1], 'recall': chosen[2], 'f1': chosen[3]}

def evaluate_model(name, model, X_tr, y_tr, X_va, y_va):
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_tr, y_tr)
    val_proba = pipe.predict_proba(X_va)[:, 1] if hasattr(pipe.named_steps['model'], 'predict_proba') else pipe.decision_function(X_va)
    threshold, thr_metrics = find_best_threshold(y_va, val_proba, min_precision=0.5)
    val_pred = (val_proba >= threshold).astype(int)
    metrics = {
        'model_name': name,
        'threshold': float(threshold),
        'f1': float(f1_score(y_va, val_pred, zero_division=0)),
        'precision': float(precision_score(y_va, val_pred, zero_division=0)),
        'recall': float(recall_score(y_va, val_pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_va, val_proba)),
        'pr_auc': float(average_precision_score(y_va, val_proba))
    }
    return pipe, metrics

models = {
    'logreg_balanced': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE),
    'random_forest_balanced': RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=2,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    ),
    'gradient_boosting': GradientBoostingClassifier(random_state=RANDOM_STATE)
}

trained_models = {}
results = []
for name, model in models.items():
    pipe, metrics = evaluate_model(name, model, X_train_fe, y_train, X_val_fe, y_val)
    trained_models[name] = pipe
    results.append(metrics)
    print(name, metrics)

results_df = pd.DataFrame(results).sort_values(['f1', 'pr_auc', 'recall'], ascending=False).reset_index(drop=True)
print(results_df)

fig = px.bar(results_df, x='model_name', y='f1', color='model_name', text='f1', title='Validation F1 by baseline model')
FIGS.append(fig)
fig.show()

Numeric cols: 55
Categorical cols: 12
logreg_balanced {'model_name': 'logreg_balanced', 'threshold': 0.9493728126710193, 'f1': 0.6538461538461539, 'precision': 0.6230366492146597, 'recall': 0.6878612716763006, 'roc_auc': 0.9708095033453991, 'pr_auc': 0.72052038592928}
random_forest_balanced {'model_name': 'random_forest_balanced', 'threshold': 0.634545840569808, 'f1': 0.7508305647840532, 'precision': 0.8828125, 'recall': 0.653179190751445, 'roc_auc': 0.9795454198012974, 'pr_auc': 0.8209430100010815}
gradient_boosting {'model_name': 'gradient_boosting', 'threshold': 0.6966059617898458, 'f1': 0.785234899328859, 'precision': 0.936, 'recall': 0.6763005780346821, 'roc_auc': 0.9695958513314501, 'pr_auc': 0.8151772451462225}
               model_name  threshold        f1  precision    recall   roc_auc    pr_auc
0       gradient_boosting   0.696606  0.785235   0.936000  0.676301  0.969596  0.815177
1  random_forest_balanced   0.634546  0.750831   0.882812  0.653179  0.979545  0.820943
2       

## Подбор гиперпараметров для лучшей baseline-модели

Тюнинг проводится только после выбора лучшей baseline-модели. Для устойчивости оптимизируем `f1` через `RandomizedSearchCV` с `StratifiedKFold`. После тюнинга отдельно подбираем рабочий threshold на hold-out validation.

In [ ]:
best_baseline_name = results_df.iloc[0]['model_name']
print('Best baseline:', best_baseline_name)

neg_pos_ratio = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
print('Train neg/pos ratio:', round(neg_pos_ratio, 4))

search_spaces = {
    'logreg_balanced': {
        'model': LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE, max_iter=4000),
        'params': {
            'model__C': np.logspace(-2, 2, 12),
            'model__solver': ['liblinear', 'lbfgs'],
            'model__penalty': ['l2']
        }
    },
    'random_forest_balanced': {
        'model': RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
        'params': {
            'model__n_estimators': [200, 300, 500, 700],
            'model__max_depth': [None, 8, 12, 16, 24],
            'model__min_samples_split': [2, 5, 10],
            'model__min_samples_leaf': [1, 2, 4],
            'model__max_features': ['sqrt', 'log2', None]
        }
    },
    'gradient_boosting': {
        'model': GradientBoostingClassifier(random_state=RANDOM_STATE),
        'params': {
            'model__n_estimators': [100, 200, 300, 500],
            'model__learning_rate': [0.02, 0.03, 0.05, 0.08, 0.1],
            'model__max_depth': [2, 3, 4, 5],
            'model__min_samples_split': [2, 5, 10],
            'model__min_samples_leaf': [1, 2, 4],
            'model__subsample': [0.7, 0.85, 1.0],
            'model__max_features': [None, 'sqrt', 'log2']
        }
    }
}

search_cfg = search_spaces[best_baseline_name]
search_pipe = Pipeline(steps=[('preprocessor', preprocessor), ('model', search_cfg['model'])])
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

random_search = RandomizedSearchCV(
    estimator=search_pipe,
    param_distributions=search_cfg['params'],
    n_iter=20,
    scoring='f1',
    n_jobs=-1,
    cv=cv,
    random_state=RANDOM_STATE,
    verbose=1
)

random_search.fit(X_train_fe, y_train)
print('Best params:', random_search.best_params_)
print('Best CV F1:', random_search.best_score_)

tuned_model = random_search.best_estimator_
val_proba_tuned = tuned_model.predict_proba(X_val_fe)[:, 1] if hasattr(tuned_model.named_steps['model'], 'predict_proba') else tuned_model.decision_function(X_val_fe)
best_threshold, threshold_info = find_best_threshold(y_val, val_proba_tuned, min_precision=0.5)
print('Best validation threshold:', best_threshold)
print('Threshold metrics:', threshold_info)
model_name_final = f'{best_baseline_name}_tuned'

Best baseline: gradient_boosting
Train neg/pos ratio: 19.6308
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best params: {'model__subsample': 0.85, 'model__n_estimators': 500, 'model__min_samples_split': 10, 'model__min_samples_leaf': 2, 'model__max_features': None, 'model__max_depth': 3, 'model__learning_rate': 0.1}
Best CV F1: 0.7725027609624814
Best validation threshold: 0.7160583343171846
Threshold metrics: {'precision': np.float64(0.9219858156028369), 'recall': np.float64(0.7514450867052023), 'f1': np.float64(0.8280254777070063)}


## Финальная модель и тест

Финальная модель переобучается на `train + val`, затем один раз оценивается на `test`. Сохраняем модель, метрики и confusion matrix.

In [ ]:
X_trainval_fe = add_features(X_train_full)
final_model = clone(tuned_model)
final_model.fit(X_trainval_fe, y_train_full)

test_proba = final_model.predict_proba(X_test_fe)[:, 1] if hasattr(final_model.named_steps['model'], 'predict_proba') else final_model.decision_function(X_test_fe)
test_pred = (test_proba >= best_threshold).astype(int)

test_metrics = {
    'f1': float(f1_score(y_test, test_pred, zero_division=0)),
    'precision': float(precision_score(y_test, test_pred, zero_division=0)),
    'recall': float(recall_score(y_test, test_pred, zero_division=0)),
    'roc_auc': float(roc_auc_score(y_test, test_proba)),
    'pr_auc': float(average_precision_score(y_test, test_proba)),
    'threshold': float(best_threshold)
}
print('Test metrics:', test_metrics)
print(classification_report(y_test, test_pred, digits=4))

joblib.dump(final_model, MODEL_PATH)
print('Saved model to:', MODEL_PATH)

cm = confusion_matrix(y_test, test_pred)
fig_cm = px.imshow(cm, text_auto=True, color_continuous_scale='Blues', title='Confusion Matrix on Test')
fig_cm.update_xaxes(title='Predicted')
fig_cm.update_yaxes(title='Actual')
FIGS.append(fig_cm)
fig_cm.show()

Test metrics: {'f1': 0.8498402555910544, 'precision': 0.95, 'recall': 0.7687861271676301, 'roc_auc': 0.9844390957315798, 'pr_auc': 0.8993570538956933, 'threshold': 0.7160583343171846}
              precision    recall  f1-score   support

           0     0.9884    0.9979    0.9931      3403
           1     0.9500    0.7688    0.8498       173

    accuracy                         0.9869      3576
   macro avg     0.9692    0.8834    0.9215      3576
weighted avg     0.9865    0.9869    0.9862      3576

Saved model to: /Users/iuriipostnii/Desktop/ГП3/gp3/data/memory/best_model.pkl


## Сравнение с предыдущим best

Если предыдущий best существует, сравниваем по `test f1`. Перезаписываем best-артефакты только если новая модель лучше. Но актуальный JSON с результатом текущего запуска всё равно формируем в требуемом формате.

In [ ]:
current_payload = {
    'model_name': model_name_final,
    'test_metrics': {
        'f1': test_metrics['f1'],
        'precision': test_metrics['precision'],
        'recall': test_metrics['recall'],
        'roc_auc': test_metrics['roc_auc'],
        'pr_auc': test_metrics['pr_auc'],
        'threshold': test_metrics['threshold']
    },
    'validation_best_threshold': float(best_threshold),
    'best_params': random_search.best_params_
}

with open(METRICS_PATH, 'w') as f:
    json.dump({'model_name': model_name_final, 'test_metrics': test_metrics}, f)

joblib.dump(final_model, MODEL_PATH)
print('Saved final artifacts to:', MODEL_PATH, METRICS_PATH)
print('Current payload:', current_payload)

Saved final artifacts to: /Users/iuriipostnii/Desktop/ГП3/gp3/data/memory/best_model.pkl /Users/iuriipostnii/Desktop/ГП3/gp3/data/memory/best_metrics.json
Current payload: {'model_name': 'gradient_boosting_tuned', 'test_metrics': {'f1': 0.8498402555910544, 'precision': 0.95, 'recall': 0.7687861271676301, 'roc_auc': 0.9844390957315798, 'pr_auc': 0.8993570538956933, 'threshold': 0.7160583343171846}, 'validation_best_threshold': 0.7160583343171846, 'best_params': {'model__subsample': 0.85, 'model__n_estimators': 500, 'model__min_samples_split': 10, 'model__min_samples_leaf': 2, 'model__max_features': None, 'model__max_depth': 3, 'model__learning_rate': 0.1}}


## Self-critique

Что можно улучшить в следующей итерации:

- добавить более сильные модели для таблично-текстовых данных: `LightGBM` / `XGBoost` с `scale_pos_weight`;
- попробовать TF-IDF только для ключевых текстовых полей (`title`, `company_profile`, `description`) в гибридном пайплайне;
- сделать более строгий out-of-fold target encoding для среднекардинальных категориальных признаков;
- провести отдельную оптимизацию threshold под конкретное бизнес-ограничение на precision;
- изучить калибровку вероятностей и error analysis по false negatives/false positives.